## ชั้นพิสูจน์ความตรง (ไม่มี ground truth ก็ตรวจได้)
- **Synthetic benchmark:** สร้างภาพต้นจำลองที่รู้ค่าจริงแน่นอน (จำนวนใบ/พื้นที่/ความสูง) → รัน SAM3 เทียบ → คำนวณ IoU/Dice/MAE = หลักฐานว่า pipeline วัดถูกต้องเชิงเทคนิค
- **Sanity check:** ดูความสัมพันธ์เชิงชีวภาพระหว่าง feature ของภาพจริง (ควรสอดคล้องกัน เช่น ใบเยอะ→coverage สูง)
- **ถ้ามี ground_truth.csv จริงทีหลัง** (image, leaf_count, shoot_count, root_count, height_cm, width_cm, area_cm2) วางใน /content/data → ระบบคำนวณ Pearson/MAE/RMSE เทียบการวัดมือให้อัตโนมัติ

## ⚙️ โหมดการประเมิน
- **ค่ากลาง (default):** verdict/readiness ใช้ threshold กลางข้ามชนิด (coverage 0.35/0.80) ใช้ได้กับหลายชนิดพืชทันที
- `species_map.csv` (image,species) — ถ้ามีจะบันทึกชนิดเป็นคอลัมน์ข้อมูลเท่านั้น (ยังไม่เปลี่ยน verdict)
- เปิด `USE_SPECIES_THRESHOLDS = True` ใน cell config เมื่อมี threshold เฉพาะชนิดจริงแล้ว


In [ ]:
# ติดตั้ง dependency — torch มี CUDA build อยู่แล้วใน Colab
!pip install -q --upgrade pip
!pip install -q transformers accelerate huggingface_hub opencv-python-headless pillow matplotlib pandas seaborn tabulate openpyxl

In [ ]:
import os, time, glob
import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'[INFO] Execution Device: {device}')
if device.type == 'cuda':
    print(f'[INFO] GPU Active: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ ต้องใช้ GPU runtime (T4) — facebook/sam3 ไม่รองรับ CPU')

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# ===== คอนฟิก — แก้ได้ตามต้องการ =====

PROMPTS = ['plant', 'leaf', 'shoot', 'stem', 'root']
# plant+shoot ใช้นับยอด/วัดโครงสร้าง | leaf นับใบ | stem/root วัดส่วนประกอบ

SCORE_THRESHOLD = 0.5
MASK_THRESHOLD = 0.5

DETECT_BOTTLE = True    # SAM3 หา 'glass jar bottle' มากำหนด ROI → coverage สัมพัทธ์ขวด
RESULTS_DIR = '/content/results'
PIXEL_TO_CM = None      # 1 px = ? cm — calibrate กับขวดจริงก่อนใช้ค่า cm

LOWER_GREEN = (35, 40, 40)
UPPER_GREEN = (85, 255, 255)
LOWER_YELLOW = (15, 60, 60)
UPPER_YELLOW = (35, 255, 255)
LOWER_BROWN = (0, 40, 40)
UPPER_BROWN = (15, 200, 130)

GLARE_V = 0.95          # pixel ที่ V เกินค่านี้ + S ต่ำกว่า GLARE_S = แสงสะท้อน
GLARE_S = 0.15
CONDENSE_V = 0.92       # ช่วงกว้างกว่า glare (ฝ้า/ไอน้ำ)
CONDENSE_S = 0.30

BASE_CONFIDENCE = 0.80
COVERAGE_READY = 0.35   # threshold กลาง (ชนิดที่ไม่มีในตารางใช้ค่านี้)
COVERAGE_OVERDENSE = 0.80

# ⚙️ การประเมินแบบค่ากลาง (generic) — ใช้ได้กับหลายชนิดพืชก่อน
# เกณฑ์ด้านล่างเป็นค่ากลางข้ามชนิด ยังไม่ผูกกับชนิดเฉพาะ
# ถ้ามีข้อมูลจริงต่อชนิดแล้ว ใส่ species_map.csv (image,species) + เปิด USE_SPECIES_THRESHOLDS
USE_SPECIES_THRESHOLDS = False  # True = ใช้ threshold ต่อชนิดจาก SPECIES_THRESHOLDS
SPECIES_THRESHOLDS = {
    'กล้วย': {'ready': 0.35, 'overdense': 0.80},
    'กล้วยไม้': {'ready': 0.30, 'overdense': 0.75},
    'มันฝรั่ง': {'ready': 0.40, 'overdense': 0.85},
}




In [ ]:
from transformers import Sam3Processor, Sam3Model

start = time.time()
model = Sam3Model.from_pretrained('facebook/sam3').to(device)
processor = Sam3Processor.from_pretrained('facebook/sam3')
print(f'โหลดโมเดลเสร็จใน {time.time() - start:.1f} วินาที')

In [ ]:
from google.colab import files as colab_files

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

def extract_zips():
    import zipfile
    for z in glob.glob(os.path.join(DATA_DIR, '*.zip')):
        print(f'พบ zip: {os.path.basename(z)} — กำลังแตก...')
        with zipfile.ZipFile(z) as zf:
            n_img = 0
            for n in zf.namelist():
                if n.lower().endswith(('.jpg', '.jpeg', '.png')):
                    try:
                        zf.extract(n, DATA_DIR)
                        n_img += 1
                    except Exception as ex:
                        print(f'ข้าม {n}: {ex}')
            print(f'แตกแล้ว {n_img} ไฟล์ภาพ')


def load_images():
    exts = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG')
    paths = []
    for e in exts:
        paths += glob.glob(os.path.join(DATA_DIR, e))
    extract_zips()
    for e in exts:
        paths += glob.glob(os.path.join(DATA_DIR, e))
    if not paths:
        print('ไม่พบภาพใน /content/data — เลือกอัปโหลด (รองรับหลายไฟล์)...')
        uploaded = colab_files.upload()
        for name, data in uploaded.items():
            if name.lower().endswith('.zip'):
                with open(os.path.join(DATA_DIR, name), 'wb') as f:
                    f.write(data)
            else:
                with open(os.path.join(DATA_DIR, name), 'wb') as f:
                    f.write(data)
        extract_zips()
        for e in exts:
            paths += glob.glob(os.path.join(DATA_DIR, e))
    images = {}
    for p in sorted(paths):
        try:
            images[os.path.basename(p)] = Image.open(p).convert('RGB')
        except Exception as ex:
            print(f'ข้ามไฟล์ {p}: {ex}')
    return images

images = load_images()
print(f'โหลดภาพ {len(images)} ภาพ')
if images:
    name, img = next(iter(images.items()))
    print(f'ตัวอย่าง: {name} — ขนาด {img.size}')

In [ ]:
# ===== ฟังก์ชันช่วย (numpy/cv2 ล้วน) =====

def masks_to_numpy(result):
    masks = result['masks']
    if hasattr(masks, 'cpu'):
        masks = masks.cpu()
    masks = np.asarray(masks).astype(bool)
    scores = result.get('scores')
    if scores is not None:
        if hasattr(scores, 'cpu'):
            scores = scores.cpu()
        scores = np.asarray(scores)
    return masks, scores

def union_bbox(mask):
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return None
    return xs.min(), ys.min(), xs.max() - xs.min() + 1, ys.max() - ys.min() + 1

def segment_prompt(image, prompt):
    inputs = processor(images=image, text=prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    result = processor.post_process_instance_segmentation(
        outputs, threshold=SCORE_THRESHOLD, mask_threshold=MASK_THRESHOLD,
        target_sizes=inputs.get('original_sizes').tolist())[0]
    return masks_to_numpy(result)

def count_confident(scores, masks):
    if scores is not None and len(scores) > 0:
        return int((scores >= SCORE_THRESHOLD).sum())
    return len(masks)

def safe_div(a, b):
    return float(a / b) if b > 0 else 0.0

def generic_species(filename):
    return 'ไม่ระบุชนิด'


In [ ]:
# ===== ฟอนต์ไทยสำหรับ matplotlib (กันตัวหนังสือเป็นกล่อง □) =====
import matplotlib.font_manager as fm

_font_cands = []
for _f in fm.findSystemFonts():
    if 'thai' in os.path.basename(_f).lower():
        _font_cands.append(_f)
for _pref in ['NotoSansThai', 'Tahoma', 'LeelawadeeUI', 'AngsanaNew', 'THSarabun']:
    for _f in fm.findSystemFonts():
        if os.path.basename(_f).startswith(_pref):
            _font_cands.append(_f)
            break
_used = False
for _f in _font_cands:
    try:
        fm.fontManager.addfont(_f)
        plt.rcParams['font.family'] = fm.FontProperties(fname=_f).get_name()
        print(f'ใช้ฟอนต์ไทย: {os.path.basename(_f)}')
        _used = True
        break
    except Exception:
        continue
if not _used:
    print('ไม่พบฟอนต์ไทย — ตัวหนังสือไทยในกราฟอาจเป็นกล่อง (ลอง pip install fonts-noto-thai)')


# ===== ดึง feature หลายมิติจาก mask + สีภาพ (ภายใน ROI) =====

def _merged_count(mask, dilate_k=7, min_area_frac=0.01):
    # นับอวัยวะ: รวมชิ้นส่วนที่ติดกัน (กัน over-segmentation) + ตัดชิ้นเล็กเกินไปทิ้ง
    if not mask.any():
        return 0, [], None
    dil = cv2.dilate(mask.astype(np.uint8), np.ones((dilate_k, dilate_k), np.uint8))
    num, labels = cv2.connectedComponents(dil)
    comps = [int((labels == i).sum()) for i in range(1, num)]
    if not comps:
        return 0, [], None
    min_area = max(200, min_area_frac * max(comps))
    areas = [a for a in comps if a >= min_area]
    merged = np.zeros_like(mask)
    for i in range(1, num):
        if comps[i - 1] >= min_area:
            merged |= (labels == i)
    return len(areas), areas, merged


def extract_features(img, masks_by_prompt, roi=None):
    rgb = np.array(img)
    H, W = rgb.shape[:2]

    if roi is None:
        roi = np.ones((H, W), dtype=bool)
    rbb = union_bbox(roi)
    if rbb is None:
        rbb = (0, 0, W, H)
    _, _, roi_w, roi_h = rbb
    roi_area = int(roi.sum())

    # --- มิติ 1: โครงสร้าง ---
    union = np.zeros((H, W), dtype=bool)
    for p in PROMPTS:
        m = masks_by_prompt.get(p, (np.zeros((0, H, W), dtype=bool), None))[0]
        if len(m) > 0:
            union |= m.any(axis=0)
    mask_in_roi = union & roi
    area = int(mask_in_roi.sum())
    coverage_ratio = safe_div(area, roi_area)

    ph_union = np.zeros((H, W), dtype=bool)
    for p in ('plant', 'shoot'):
        m = masks_by_prompt.get(p, (np.zeros((0, H, W), dtype=bool), None))[0]
        if len(m) > 0:
            ph_union |= m.any(axis=0)
    if not ph_union.any():
        ph_union = mask_in_roi
    bb = union_bbox(ph_union & roi)
    if bb is not None:
        _, _, bw, bh = bb
        height_proxy = safe_div(bh, roi_h)
        width_proxy = safe_div(bw, roi_w)
        aspect_ratio = safe_div(bh, bw)
        compactness = safe_div(area, bh * bw)
    else:
        height_proxy = width_proxy = aspect_ratio = compactness = 0.0
        bw = bh = 0

    # --- มิติ 3: ความซับซ้อนขอบ ---
    hull_ratio = 0.0
    perimeter_px = 0.0
    if area > 0:
        contours, _ = cv2.findContours((mask_in_roi.astype(np.uint8)) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            cnt = max(contours, key=cv2.contourArea)
            hull_ratio = safe_div(area, cv2.contourArea(cv2.convexHull(cnt)))
            perimeter_px = float(cv2.arcLength(cnt, True))
    perimeter_ratio = safe_div(perimeter_px, 2 * (roi_w + roi_h))

    # --- มิติ 2: จำนวนอวัยวะ ---
    counts = {}
    for p in PROMPTS:
        m, s = masks_by_prompt.get(p, (np.zeros((0, H, W), dtype=bool), None))
        counts[p] = count_confident(s, m)

    leaf_masks = masks_by_prompt.get('leaf', (np.zeros((0, H, W), dtype=bool), None))[0]
    leaf_mask_union = leaf_masks.any(axis=0) if len(leaf_masks) > 0 else None
    leaf_count_raw = counts['leaf']
    leaf_count_method = 'prompt'
    if leaf_mask_union is None or not leaf_mask_union.any():
        leaf_mask_union = (ph_union & roi) if ph_union.any() else None
        if leaf_mask_union is not None:
            leaf_count_method = 'fallback'
    if leaf_mask_union is not None and leaf_mask_union.any():
        leaf_count, leaf_areas, leaf_merged = _merged_count(leaf_mask_union & roi)
    else:
        leaf_count, leaf_areas = 0, []
    mean_leaf_area = float(np.mean(leaf_areas)) if leaf_areas else 0.0
    leaf_area_cv = safe_div(float(np.std(leaf_areas)), max(np.mean(leaf_areas), 1e-6)) if leaf_areas else 0.0
    max_leaf_area = float(max(leaf_areas)) if leaf_areas else 0.0

    # --- มิติ 4: สี/สุขภาพ (ภายใน mask ใน ROI) ---
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    green = cv2.inRange(hsv, np.array(LOWER_GREEN), np.array(UPPER_GREEN)) > 0
    yellow = cv2.inRange(hsv, np.array(LOWER_YELLOW), np.array(UPPER_YELLOW)) > 0
    brown = cv2.inRange(hsv, np.array(LOWER_BROWN), np.array(UPPER_BROWN)) > 0
    if area > 0:
        green_pct = 100.0 * int((green & mask_in_roi).sum()) / area
        yellow_pct = 100.0 * int((yellow & mask_in_roi).sum()) / area
        brown_pct = 100.0 * int((brown & mask_in_roi).sum()) / area
        dark_green_ratio = 100.0 * int(((hsv[..., 2] < 90) & green & mask_in_roi).sum()) / area
        g_ratio = rgb[..., 1].astype(np.float32) / (rgb.sum(axis=2).astype(np.float32) + 1e-6)
        greenness = float(g_ratio[mask_in_roi].mean())
        mean_hue = float(hsv[..., 0][mask_in_roi].mean())
        mean_sat = float(hsv[..., 1][mask_in_roi].mean())
        mean_val = float(hsv[..., 2][mask_in_roi].mean())
    else:
        green_pct = yellow_pct = brown_pct = dark_green_ratio = 0.0
        greenness = mean_hue = mean_sat = mean_val = 0.0
    healthy_color = green_pct - yellow_pct - brown_pct

    # --- มิติ 5: คุณภาพภาพ (ทั้ง ROI) ---
    v = hsv[..., 2].astype(np.float32) / 255.0
    s = hsv[..., 1].astype(np.float32) / 255.0
    glare_score = 100.0 * int(((v > GLARE_V) & (s < GLARE_S) & roi).sum()) / max(roi_area, 1)
    condensation_score = 100.0 * int(((v > CONDENSE_V) & (s < CONDENSE_S) & roi).sum()) / max(roi_area, 1)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)[roi]
    brightness_mean = float(gray.mean()) if len(gray) else 0.0
    brightness_std = float(gray.std()) if len(gray) else 0.0

    # --- confidence รวมทุก prompt ---
    score_list = [s for _, s in masks_by_prompt.values() if s is not None and len(s) > 0]
    if score_list:
        all_scores = np.concatenate(score_list)
        mean_score = float(all_scores.mean())
        min_score = float(all_scores.min())
        score_std = float(all_scores.std())
    else:
        mean_score = min_score = score_std = 0.0

    return {
        'total_area_px': area,
        'coverage_ratio': round(coverage_ratio, 6),
        'height_proxy': round(height_proxy, 6),
        'width_proxy': round(width_proxy, 6),
        'aspect_ratio': round(aspect_ratio, 6),
        'compactness': round(compactness, 6),
        'canopy_h_cm': round(bh * PIXEL_TO_CM, 2) if (bb is not None and PIXEL_TO_CM) else None,
        'canopy_w_cm': round(bw * PIXEL_TO_CM, 2) if (bb is not None and PIXEL_TO_CM) else None,
        'hull_ratio': round(hull_ratio, 6),
        'perimeter_px': round(perimeter_px, 2),
        'perimeter_ratio': round(perimeter_ratio, 6),
        'leaf_count': leaf_count,
        'leaf_count_raw': leaf_count_raw,
        'leaf_count_method': leaf_count_method,
        'shoot_count': counts['plant'] + counts['shoot'],
        'root_count': counts['root'],
        'stem_count': counts['stem'],
        'mean_leaf_area_px': round(mean_leaf_area, 2),
        'leaf_area_cv': round(leaf_area_cv, 4),
        'max_leaf_area_px': round(max_leaf_area, 2),
        'green_pct': round(green_pct, 2),
        'greenness': round(greenness, 4),
        'dark_green_ratio': round(dark_green_ratio, 2),
        'yellow_ratio': round(yellow_pct, 2),
        'brown_ratio': round(brown_pct, 2),
        'healthy_color': round(healthy_color, 2),
        'mean_hue': round(mean_hue, 2),
        'mean_sat': round(mean_sat, 2),
        'mean_val': round(mean_val, 2),
        'glare_score': round(glare_score, 2),
        'condensation_score': round(condensation_score, 2),
        'brightness_mean': round(brightness_mean, 2),
        'brightness_std': round(brightness_std, 2),
        'mean_score': round(mean_score, 4),
        'min_score': round(min_score, 4),
        'score_std': round(score_std, 4),
    }


In [ ]:
# ===== วิเคราะห์ 1 ภาพ: segment + feature + species + verdict + confidence =====

def analyze_image(img, filename, species=None):
    masks_by_prompt = {}
    roi = None
    if DETECT_BOTTLE:
        b_masks, _ = segment_prompt(img, 'glass jar bottle')
        if len(b_masks) > 0:
            bb = union_bbox(b_masks.any(axis=0))
            if bb is not None:
                x, y, bw, bh = bb
                roi = np.zeros((np.array(img).shape[0], np.array(img).shape[1]), dtype=bool)
                roi[y:y + bh, x:x + bw] = True
    for prompt in PROMPTS:
        masks_by_prompt[prompt] = segment_prompt(img, prompt)

    feat = extract_features(img, masks_by_prompt, roi)
    if species is None:
        species = generic_species(filename)
    if USE_SPECIES_THRESHOLDS and species in SPECIES_THRESHOLDS:
        th = SPECIES_THRESHOLDS[species]
    else:
        th = {'ready': COVERAGE_READY, 'overdense': COVERAGE_OVERDENSE}  # ค่ากลาง generic

    if feat['coverage_ratio'] >= th['overdense']:
        verdict = 'หนาแน่นเกิน-ตรวจ'
    elif feat['coverage_ratio'] >= th['ready']:
        verdict = 'พร้อมอนุบาล'
    else:
        verdict = 'ยังไม่พร้อม'

    confidence = max(BASE_CONFIDENCE * (1.0 - feat['glare_score'] / 100.0), 0.0)
    readiness_index = (0.4 * min(feat['coverage_ratio'] / max(th['overdense'], 1e-6), 1.0)
                       + 0.3 * min(feat['height_proxy'], 1.0)
                       + 0.3 * min(feat['green_pct'] / 100.0, 1.0))

    notes = []
    if DETECT_BOTTLE and roi is None:
        notes.append('ไม่พบขวด (SAM3) → ROI ทั้งภาพ')
    if feat['glare_score'] > 40:
        notes.append('glare สูง confidence ถูกลด')
    if feat['condensation_score'] > 40:
        notes.append('ภาพมัว/ฝ้า ตรวจผลระวัง')
    if feat['leaf_count_method'] == 'fallback':
        notes.append('leaf prompt ไม่เจอ → นับใบจาก plant+shoot')
    if feat['leaf_count'] == 0 and feat['coverage_ratio'] > 0.05:
        notes.append('มี coverage แต่ไม่พบใบ')
    if feat['yellow_ratio'] > 30:
        notes.append('ใบเหลืองมาก')

    feat.update({
        'image': filename,
        'species': species,
        'verdict': verdict,
        'readiness_index': round(readiness_index, 4),
        'confidence': round(confidence, 4),
        'note': ' | '.join(notes),
    })
    return feat, masks_by_prompt


In [ ]:
# ===== รันวิเคราะห์ทุกภาพ =====
SPECIES_MAP = {}
sp_csv = os.path.join(DATA_DIR, 'species_map.csv')
if os.path.exists(sp_csv):
    _sp = pd.read_csv(sp_csv)
    SPECIES_MAP = dict(zip(_sp['image'].astype(str), _sp['species'].astype(str)))
    print(f'โหลด species_map.csv — {len(SPECIES_MAP)} รายการ (ภาพที่ไม่อยู่ในตารางใช้ auto-match)')
else:
    print('ไม่มี species_map.csv — species เป็น "ไม่ระบุชนิด" (ใช้เกณฑ์ค่ากลางกับทุกภาพ)')

rows = []
all_masks = {}
total = len(images) * (len(PROMPTS) + (1 if DETECT_BOTTLE else 0))
job = 0

for name, img in images.items():
    feat, mbp = analyze_image(img, name, species=SPECIES_MAP.get(name))
    all_masks[name] = mbp
    rows.append(feat)
    job += len(PROMPTS)
    print(f'[{job}/{total}] {name} — leaf={feat["leaf_count"]} shoot={feat["shoot_count"]} '
          f'cov={feat["coverage_ratio"]:.2f} green={feat["green_pct"]:.0f}% verdict={feat["verdict"]}')

df = pd.DataFrame(rows)
print(f'รวม {len(df)} ภาพ × {len(PROMPTS)} prompts → {df.shape[1]} คอลัมน์ feature')

In [ ]:
# แสดงตารางรวม (เฉพาะคอลัมน์หลัก)
show_cols = ['image', 'species', 'leaf_count', 'shoot_count', 'root_count', 'stem_count',
             'coverage_ratio', 'height_proxy', 'green_pct', 'yellow_ratio', 'hull_ratio',
             'readiness_index', 'confidence', 'verdict', 'note']
display(df[show_cols])

In [ ]:
# ===== วาด overlay ทุก prompt (6 ภาพแรก) + บันทึก PNG =====
import os

os.makedirs(os.path.join(RESULTS_DIR, 'overlays'), exist_ok=True)

def draw_overlay(img, masks, color=(0, 200, 0)):
    img_rgb = np.array(img).copy()
    for m in masks:
        contours, _ = cv2.findContours((m.astype(np.uint8)) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(img_rgb, contours, -1, color, 2)
    return img_rgb

COLORS = {'plant': (0, 200, 0), 'leaf': (0, 120, 255), 'shoot': (255, 120, 0),
          'stem': (120, 60, 200), 'root': (200, 0, 200)}

overlay_paths = []
n_overlay = min(6, len(images))
for k, name in enumerate(list(images)[:n_overlay]):
    n = 1 + len(PROMPTS)
    fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4.2))
    axes[0].imshow(images[name])
    axes[0].set_title('ต้นฉบับ')
    axes[0].axis('off')
    for i, p in enumerate(PROMPTS):
        masks, _ = all_masks[name][p]
        ov = draw_overlay(images[name], masks, COLORS.get(p, (0, 200, 0)))
        axes[i + 1].imshow(ov)
        axes[i + 1].set_title(f'{p} — {len(masks)} mask')
        axes[i + 1].axis('off')
    plt.suptitle(name, fontsize=12)
    plt.tight_layout()
    png = os.path.join(RESULTS_DIR, 'overlays', f'overlay_{k + 1:02d}_{name.replace(chr(46), chr(95))}.png')
    fig.savefig(png, dpi=120, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    overlay_paths.append(png)

print(f'บันทึก overlay: {len(overlay_paths)} ภาพ → {RESULTS_DIR}/overlays/')


In [ ]:
# ===== กราฟสรุป: radar + %green + scatter + heatmap + pie/hist (บันทึก PNG) =====
import math

plot_dir = os.path.join(RESULTS_DIR, 'plots')
os.makedirs(plot_dir, exist_ok=True)
plot_paths = []

dims = pd.DataFrame(index=df['image'])
dims['โครงสร้าง'] = (df['coverage_ratio'] + df['height_proxy']) / 2
dims['ความซับซ้อน'] = df['hull_ratio']
dims['สีสุขภาพ'] = df['green_pct'] / 100.0
dims['อวัยวะ'] = np.clip((df['leaf_count'] + df['shoot_count']) / 10.0, 0, 1)
mean_dims = dims.mean()

labels = list(mean_dims.index)
values = list(mean_dims.values) + [list(mean_dims.values)[0]]
angles = [i * 2 * math.pi / len(labels) for i in range(len(labels))] + [0]

fig1 = plt.figure(figsize=(6, 6))
ax1 = fig1.add_subplot(111, polar=True)
ax1.plot(angles, values, 'o-', linewidth=2, color='seagreen')
ax1.fill(angles, values, alpha=0.25, color='seagreen')
ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(labels)
ax1.set_ylim(0, 1)
ax1.set_title('ค่าเฉลี่ย 4 มิติของทุกภาพ (0-1)')
plt.tight_layout()
p1 = os.path.join(plot_dir, 'radar_4dim.png')
fig1.savefig(p1, dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig1)
plot_paths.append(p1)

fig2, axes2 = plt.subplots(1, 2, figsize=(13, 5))
axes2[0].bar(df['image'], df['green_pct'], color='seagreen')
axes2[0].set_title('Plant Health Index (% Green ภายใน mask)')
axes2[0].set_ylabel('%')
axes2[0].set_ylim(0, 100)
axes2[0].tick_params(axis='x', rotation=60, labelsize=7)
axes2[1].scatter(df['leaf_count'], df['green_pct'], s=60, color='green')
axes2[1].set_xlabel('Leaf Count (SAM3)')
axes2[1].set_ylabel('Green Coverage (%)')
axes2[1].set_title('Leaf Count vs Green')
plt.tight_layout()
p2 = os.path.join(plot_dir, 'green_bar_scatter.png')
fig2.savefig(p2, dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig2)
plot_paths.append(p2)

sel = ['coverage_ratio', 'height_proxy', 'width_proxy', 'compactness', 'leaf_count',
       'shoot_count', 'root_count', 'green_pct', 'yellow_ratio', 'hull_ratio', 'readiness_index']
cm = df[sel].corr()
fig3, ax3 = plt.subplots(figsize=(10, 8))
im = ax3.imshow(cm, cmap='coolwarm', vmin=-1, vmax=1)
ax3.set_xticks(range(len(cm)))
ax3.set_xticklabels(cm.columns, rotation=45, ha='right')
ax3.set_yticks(range(len(cm)))
ax3.set_yticklabels(cm.columns)
for i in range(len(cm)):
    for j in range(len(cm)):
        ax3.text(j, i, f'{cm.iloc[i, j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im)
ax3.set_title('Correlation ระหว่าง feature')
plt.tight_layout()
p3 = os.path.join(plot_dir, 'feature_heatmap.png')
fig3.savefig(p3, dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig3)
plot_paths.append(p3)

fig4, axes4 = plt.subplots(1, 2, figsize=(13, 5))
vc = df['verdict'].value_counts()
axes4[0].pie(vc.values, labels=vc.index, autopct='%1.0f%%', startangle=90,
             colors=['#d9534f', '#5cb85c', '#f0ad4e'])
axes4[0].set_title('สัดส่วน verdict')
axes4[1].hist(df['readiness_index'], bins=15, color='steelblue', edgecolor='white')
axes4[1].axvline(0.5, color='red', ls='--', lw=1.5)
axes4[1].set_title('การกระจาย readiness_index (เส้นแดง = 0.5)')
axes4[1].set_xlabel('readiness_index')
plt.tight_layout()
p4 = os.path.join(plot_dir, 'verdict_pie_hist.png')
fig4.savefig(p4, dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig4)
plot_paths.append(p4)

print(f'บันทึกกราฟ: {len(plot_paths)} รูป → {RESULTS_DIR}/plots/')


In [ ]:
# ===== สร้างภาพต้นจำลอง (synthetic) — รู้ค่าจริงแน่นอน =====
# วาดต้น: ก้าน + ใบ n ใบ (ตำแหน่ง/ขนาดสุ่มแต่รู้ค่า) → บันทึก GT mask เอาไว้เทียบ IoU/Dice
import random

def draw_synthetic_plant(n_leaves, H=400, W=400, seed=0):
    rng = np.random.default_rng(seed)
    canvas = np.full((H, W, 3), 250, dtype=np.uint8)
    gt_mask = np.zeros((H, W), dtype=bool)
    base_y = H - 60
    cv2.line(canvas, (W // 2, base_y), (W // 2, 120), (40, 110, 40), 6)
    gt_mask[110:base_y + 3, W // 2 - 3:W // 2 + 4] = True
    for i in range(n_leaves):
        cx = int(W // 2 + rng.integers(-90, 90))
        cy = int(130 + rng.integers(0, 190))
        rx = int(rng.integers(20, 45))
        ry = int(rng.integers(14, 30))
        angle = float(rng.integers(0, 180))
        col = (int(rng.integers(30, 60)), int(rng.integers(140, 190)), int(rng.integers(30, 60)))
        em = np.zeros((H, W), dtype=np.uint8)
        cv2.ellipse(em, (cx, cy), (rx, ry), angle, 0, 360, 255, -1)
        canvas[em > 0] = col
        gt_mask |= em > 0
    return Image.fromarray(canvas), gt_mask

SYN_DIR = os.path.join(DATA_DIR, 'synthetic')
os.makedirs(SYN_DIR, exist_ok=True)
syn_rows = []
for i, nl in enumerate([3, 5, 7, 9, 12]):
    img, gt = draw_synthetic_plant(nl, seed=i * 7 + 1)
    name = f'synth_{i:03d}_leaf{nl}.png'
    img.save(os.path.join(SYN_DIR, name))
    ys, xs = np.where(gt)
    syn_rows.append({'image': name, 'true_leaf_count': nl,
                     'true_area_px': int(gt.sum()),
                     'true_height_px': int(ys.max() - ys.min()) + 1})
    print(f'สร้าง {name} — ใบ {nl} ใบ')
syn_gt = pd.DataFrame(syn_rows)
syn_gt.to_csv(os.path.join(DATA_DIR, 'synthetic_truth.csv'), index=False, encoding='utf-8-sig')
print('ตารางค่าจริง: synthetic_truth.csv')

In [ ]:
# ===== Synthetic benchmark: รัน SAM3 บนภาพจำลองแล้วเทียบกับค่าจริง =====
syn_images = {}
for p in glob.glob(os.path.join(SYN_DIR, '*')):
    syn_images[os.path.basename(p)] = Image.open(p).convert('RGB')

syn_rows_out = []
syn_masks_out = {}
for name, img in syn_images.items():
    feat, mbp = analyze_image(img, name)
    syn_rows_out.append(feat)
    syn_masks_out[name] = mbp
syn_df = pd.DataFrame(syn_rows_out)
bm = syn_df.merge(syn_gt, on='image', suffixes=('_sam3', '_true'))

# 1) ความแม่นยำการนับใบ
bm['leaf_err'] = (bm['leaf_count'] - bm['true_leaf_count']).abs()
count_mae = float(bm['leaf_err'].mean())
count_rmse = float((bm['leaf_err'] ** 2).mean() ** 0.5)
print('=== 1) ความแม่นยำการนับใบ ===')
print(bm[['image', 'true_leaf_count', 'leaf_count', 'leaf_err']].to_string(index=False))
print(f'MAE = {count_mae:.2f} ใบ   RMSE = {count_rmse:.2f} ใบ')

# 2) IoU + Dice ระหว่าง mask ของ SAM3 (union plant+leaf) กับ GT mask จริง
ious, dices = [], []
for _, r in bm.iterrows():
    name = r['image']
    union = np.zeros_like(np.array(syn_images[name]), dtype=bool)[:, :, 0]
    for p in ('plant', 'leaf'):
        m = syn_masks_out[name].get(p, (np.zeros((0, union.shape[0], union.shape[1]), dtype=bool), None))[0]
        if len(m) > 0:
            union |= m.any(axis=0)
    gt_path = os.path.join(SYN_DIR, name)
    # เปิดภาพจำลองอีกครั้งเพื่อหา GT mask (สร้างใหม่ด้วย seed เดิม)
    n_leaf = int(r['true_leaf_count'])
    seed_idx = bm.index.get_loc(r.name)
    _, gt = draw_synthetic_plant(n_leaf, seed=seed_idx * 7 + 1)
    inter = int((union & gt).sum())
    union_all = int((union | gt).sum())
    iou = inter / max(union_all, 1)
    dice = 2 * inter / max(int(union.sum()) + int(gt.sum()), 1)
    ious.append(iou)
    dices.append(dice)
print('=== 2) IoU / Dice ของ mask เทียบ GT จริง ===')
print(f'mIoU = {np.mean(ious):.3f}   mDice = {np.mean(dices):.3f}')

# 3) พื้นที่: อัตราส่วนพื้นที่ SAM3 / พื้นที่จริง
bm['area_ratio'] = bm['total_area_px'] / bm['true_area_px'].clip(lower=1)
print('=== 3) อัตราส่วนพื้นที่ (SAM3/จริง — ใกล้ 1 = ดี) ===')
print(bm[['image', 'true_area_px', 'total_area_px', 'area_ratio']].to_string(index=False))
print('mean ratio =', round(float(bm['area_ratio'].mean()), 3))

In [ ]:
# ===== Sanity check บนภาพจริง: ความสัมพันธ์เชิงชีวภาพควรเป็นบวก/สมเหตุผล =====
pairs = [('leaf_count', 'coverage_ratio'), ('leaf_count', 'shoot_count'),
         ('coverage_ratio', 'height_proxy'), ('green_pct', 'healthy_color'),
         ('coverage_ratio', 'total_area_px')]
print('=== ความสัมพันธ์ (Pearson r) ระหว่าง feature — ค่าบวกสมเหตุผล = ค่าจากภาพสอดคล้องกัน ===')
for a, b in pairs:
    r = df[a].astype(float).corr(df[b].astype(float))
    print(f'  {a} ~ {b}: r = {r:.3f}')
print()
print('=== สถิติรวมของชุดภาพจริง ===')
print(df[['leaf_count', 'shoot_count', 'root_count', 'coverage_ratio',
          'height_proxy', 'green_pct', 'yellow_ratio', 'glare_score']].describe().round(3).to_string())

In [ ]:
# ===== Export: CSV + Excel + รายงาน HTML (รูป+กราฟฝังในไฟล์) + ZIP =====
import base64
import shutil
import time

os.makedirs(RESULTS_DIR, exist_ok=True)
csv_path = os.path.join(RESULTS_DIR, 'plant_growth_summary.csv')
xlsx_path = os.path.join(RESULTS_DIR, 'plant_growth_summary.xlsx')
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
df.to_excel(xlsx_path, index=False)


def img_b64(path):
    with open(path, 'rb') as f:
        return 'data:image/png;base64,' + base64.b64encode(f.read()).decode()


main_cols = ['image', 'verdict', 'readiness_index', 'coverage_ratio', 'leaf_count',
             'shoot_count', 'green_pct', 'yellow_ratio', 'confidence', 'note']
thead = '<tr><th>#</th>' + ''.join(f'<th>{c}</th>' for c in main_cols) + '</tr>'
trows = ''
for k, (_, r) in enumerate(df.sort_values('readiness_index', ascending=False).iterrows(), 1):
    trows += '<tr><td>{}</td>{}</tr>'.format(
        k, ''.join('<td>{}</td>'.format('' if pd.isna(r[c]) else r[c]) for c in main_cols))

vc = df['verdict'].value_counts().to_dict()
html = (
    '<html><head><meta charset="utf-8"><title>รายงานวิเคราะห์การเจริญพืช (SAM3)</title>'
    '<style>body{font-family:Tahoma,sans-serif;margin:30px}h1{color:#1a7a4f}'
    'table{border-collapse:collapse}td,th{border:1px solid #999;padding:3px 8px;font-size:12px}'
    'img{max-width:100%;border:1px solid #ccc;margin:6px 0}'
    'h2{border-bottom:2px solid #1a7a4f;padding-bottom:4px;margin-top:40px}</style></head><body>'
    f'<h1>รายงานวิเคราะห์การเจริญพืชเพาะเลี้ยงเนื้อเยื่อ (SAM3)</h1>'
    f'<p>จำนวนภาพ: {len(df)} | สร้างเมื่อ: {time.strftime(chr(37) + "Y-" + chr(37) + "m-" + chr(37) + "d " + chr(37) + "H:" + chr(37) + "M:" + chr(37) + "S")}</p>'
    f'<p><b>verdict:</b> {vc}</p>'
    '<table>' + thead + trows + '</table>'
    '<h2>ภาพ overlay ตัวอย่าง</h2>'
    + ''.join(f'<p><b>{os.path.basename(p)}</b></p><img src="{img_b64(p)}">' for p in overlay_paths)
    + '<h2>กราฟวิเคราะห์</h2>'
    + ''.join(f'<img src="{img_b64(p)}">' for p in plot_paths)
    + '</body></html>'
)
report_path = os.path.join(RESULTS_DIR, 'report.html')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(html)

zip_path = '/content/vitro_report.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive(zip_path[:-4], 'zip', RESULTS_DIR)
print(f'บันทึก: {RESULTS_DIR}/ (CSV/XLSX/report.html/overlays/plots)')
print(f'ZIP ทั้งหมด: {zip_path}')
colab_files.download(csv_path)
colab_files.download(xlsx_path)
colab_files.download(zip_path)
